- ### numeric

1. [Accumulate](#accumulate)

2. [GCD & LCM](#gcd--lcm)

---

- ### Accumulate

In [2]:
# Setup for oneline command %%cpp
import os, tempfile, subprocess
from IPython.core.magic import register_cell_magic # type: ignore
import shlex

@register_cell_magic
def cpp(line, cell):
    """
    Usage:
    %%cpp -i "input for cin" -- arg1 arg2 ...
    """
    tokens = shlex.split(line)
    input_data = None
    run_args = []

    # Parse stdin input
    if "-i" in tokens:
        idx = tokens.index("-i")
        if idx + 1 < len(tokens):
            input_data = tokens[idx + 1]

    # Parse program arguments after --
    if "--" in tokens:
        idx = tokens.index("--")
        run_args = tokens[idx + 1:]

    # Write temp C++ file
    with tempfile.NamedTemporaryFile(suffix=".cpp", delete=False, mode="w") as tmp_cpp:
        tmp_cpp.write(cell)
        cpp_path = tmp_cpp.name
    exe_path = cpp_path[:-4] + ".exe"

    try:
        # Compile
        compile_proc = subprocess.run(
            ["g++", "-std=c++23", "-O2", "-Wall", cpp_path, "-o", exe_path],
            capture_output=True,
            text=True
        )
        if compile_proc.returncode != 0:
            print("❌ Compilation failed:\n", compile_proc.stderr)
            return

        # Run program
        run_proc = subprocess.run(
            [exe_path] + run_args,
            input=input_data,      # feed stdin here
            capture_output=True,
            text=True
        )
        if run_proc.stdout:
            print(run_proc.stdout, end="")
        if run_proc.stderr:
            print("⚠️ Runtime error:\n", run_proc.stderr)

    finally:
        for f in (cpp_path, exe_path):
            try: os.remove(f)
            except: pass

**accumulate**

In [ ]:
%%cpp
#include <iostream>
#include <numeric>
#include <vector>
using namespace std;

int main() {
    vector<int> v = {5, 2, 9, 1, 5, 6};
    int sum = accumulate(v.begin(), v.end(), 0); // default operator is plus
    cout << "Sum: " << sum << endl;
}

Sum: 28


In [6]:
%%cpp
#include <iostream>
#include <numeric>
#include <vector>
using namespace std;

int main() {
    vector<int> v = {5, 2, 9, 1, 5, 6};
    int product = accumulate(v.begin(), v.end(), 1, multiplies<int>()); // operator is multiplies
    cout << "Product: " << product << endl;
}

Product: 2700


**inner_product**

In [10]:
%%cpp
#include <iostream>
#include <numeric>
#include <vector>
using namespace std;

int main() {
    vector<int> v = {5, 2, 9, 1, 5, 6};
    vector<int> u = {-5, -2, 9, 1, 5, 6};
    int dot = inner_product(v.begin(), v.end(), u.begin(), 1);
    cout << "inner_product: " << dot << endl;
}

inner_product: 115


**range**

In [16]:
%%cpp
#include <iostream>
#include <numeric>
#include <vector>
using namespace std;

int main() {
    vector<int> v(5);
    iota(v.begin(), v.end(), 10); // fills with 10, 11, 12, 13, 14
    for (int n : v) {
        cout << n << " ";
    }
}

10 11 12 13 14 

**partial_sum**

In [18]:
%%cpp
#include <iostream>
#include <numeric>
#include <vector>
using namespace std;

int main() {
    vector<int> v = {5, 2, 9, 1, 5, 6};
    vector<int> u(9);
    partial_sum(v.begin(), v.end(), u.begin());
    for (int n : u) {
        cout << n << " ";
    }
}

5 7 16 17 22 28 0 0 0 

**adjacent_difference**

In [20]:
%%cpp
#include <iostream>
#include <numeric>
#include <vector>
using namespace std;

int main() {
    vector<int> v = {5, 2, 9, 1, 5, 6};
    vector<int> u(v.size());
    adjacent_difference(v.begin(), v.end(), u.begin());
    for (int n : u) {
        cout << n << " ";
    }
}

5 -3 7 -8 4 1 

---

- ### GCD & LCM

**gcd**

In [26]:
%%cpp
#include <iostream>
#include <numeric>
#include <vector>
using namespace std;

int main() {
    vector<int> v = {3, 12, 9};
    int GCD = gcd(v[0], v[1]);
    cout << GCD << endl;
}

3


In [31]:
%%cpp
#include <iostream>
#include <numeric>
#include <vector>
using namespace std;

int GCD(vector<int> v) {
        if (v.empty()) return 0;
        return accumulate(v.begin() + 1, v.end(), v[0], [](int a, int b){return gcd(a, b);});
}
int main() {
    vector<int> v = {3, 12, 9, 6};
    cout << GCD(v) << endl;
}

3


**lcm**

In [32]:
%%cpp
#include <iostream>
#include <numeric>
#include <vector>
using namespace std;

int LCM(vector<int> v) {
        if (v.empty()) return 0;
        return accumulate(v.begin() + 1, v.end(), v[0], [](int a, int b){return lcm(a, b);});
}
int main() {
    vector<int> v = {3, 12, 9, 6};
    cout << LCM(v) << endl;
}

36
